# 📱💰 The $145 Billion Paradox

## How Mobile Video Advertising Became an Automated Empire — While Attention Fragmented

---

**A Data Story by HackrLife** | January 2026

---

### The Thesis

> *"Mobile video advertising became a $145B automated, app-dominated machine — but while the money concentrated, the audience fragmented, and YouTube lost half its attention share to TikTok, Reels, and streaming apps."*

---

### Data Sources

This analysis combines **5 EMARKETER datasets**:

1. Mobile Video Ad Spending ($)
2. Mobile Video Ad Spending Growth (%)
3. Programmatic Share (%)
4. In-App Share (%)
5. YouTube Time Share (%)

---

## 🔧 Setup & Configuration

In [ ]:
# Enable inline plotting
%matplotlib inline

# Core Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.gridspec import GridSpec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Dark Theme Configuration
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.facecolor'] = '#0a0a0a'
plt.rcParams['axes.facecolor'] = '#0a0a0a'
plt.rcParams['axes.edgecolor'] = '#333333'
plt.rcParams['axes.labelcolor'] = '#ffffff'
plt.rcParams['text.color'] = '#ffffff'
plt.rcParams['xtick.color'] = '#888888'
plt.rcParams['ytick.color'] = '#888888'
plt.rcParams['grid.color'] = '#222222'
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.size'] = 11

# Custom Color Palette
COLORS = {
    'primary': '#00D4FF',      # Cyan
    'secondary': '#FF6B6B',    # Coral
    'accent': '#4ECDC4',       # Teal
    'warning': '#FFE66D',      # Yellow
    'purple': '#A855F7',       # Purple
    'orange': '#FB923C',       # Orange
    'green': '#22C55E',        # Green
    'pink': '#EC4899',         # Pink
}

print("✅ Libraries loaded")
print("🎨 Dark theme configured")

---

## 📊 Data Loading

In [ ]:
def parse_value(val):
    """Convert string values like '$63.8M', '$1.4B', '140.1%' to numeric"""
    if pd.isna(val) or val == '':
        return np.nan
    val = str(val).strip()
    
    if '%' in val:
        return float(val.replace('%', ''))
    
    val = val.replace('$', '').replace(',', '')
    
    if 'B' in val:
        return float(val.replace('B', '')) * 1000  # Convert to millions
    elif 'M' in val:
        return float(val.replace('M', ''))
    
    try:
        return float(val)
    except:
        return np.nan

def load_emarketer_data(filepath):
    """Load and parse EMARKETER CSV files"""
    df = pd.read_csv(filepath)
    metric_name = df['Metric'].iloc[0]
    year_cols = [col for col in df.columns if col.isdigit() or (col.startswith('20') and len(col) == 4)]
    data = {}
    for year in year_cols:
        data[int(year)] = parse_value(df[year].iloc[0])
    return pd.Series(data, name=metric_name)

# Load all datasets
spending = load_emarketer_data('EMARKETER_table_330.csv')
growth = load_emarketer_data('EMARKETER_table_332.csv')
programmatic_pct = load_emarketer_data('EMARKETER_table_8732.csv')
in_app_pct = load_emarketer_data('EMARKETER_table_9669.csv')
youtube_time_pct = load_emarketer_data('EMARKETER_table_16381.csv')

print("📊 Data Loaded:")
print(f"   • Mobile Video Ad Spending: {len(spending)} years (2011-2029)")
print(f"   • Growth Rates: {len(growth)} years")
print(f"   • Programmatic %: {len(programmatic_pct.dropna())} years")
print(f"   • In-App %: {len(in_app_pct.dropna())} years")
print(f"   • YouTube Time %: {len(youtube_time_pct.dropna())} years")

---

## 📐 Create Master Dataset with Derived Metrics

In [ ]:
# Create Master DataFrame
years = list(range(2011, 2030))

df = pd.DataFrame({
    'year': years,
    'spending_millions': [spending.get(y, np.nan) for y in years],
    'growth_pct': [growth.get(y, np.nan) for y in years],
    'programmatic_pct': [programmatic_pct.get(y, np.nan) for y in years],
    'in_app_pct': [in_app_pct.get(y, np.nan) for y in years],
    'youtube_time_pct': [youtube_time_pct.get(y, np.nan) for y in years]
})

df.set_index('year', inplace=True)

# DERIVED METRICS
df['programmatic_dollars'] = df['spending_millions'] * (df['programmatic_pct'] / 100)
df['manual_dollars'] = df['spending_millions'] - df['programmatic_dollars']
df['in_app_dollars'] = df['spending_millions'] * (df['in_app_pct'] / 100)
df['mobile_web_dollars'] = df['spending_millions'] - df['in_app_dollars']
df['youtube_implied_opportunity'] = df['spending_millions'] * (df['youtube_time_pct'] / 100)
df['non_youtube_opportunity'] = df['spending_millions'] - df['youtube_implied_opportunity']
df['automation_velocity'] = df['programmatic_pct'].diff()
df['dollars_per_yt_attention_point'] = df['spending_millions'] / df['youtube_time_pct']

def classify_era(year):
    if year <= 2016:
        return 'Act 1: Wild West'
    elif year <= 2022:
        return 'Act 2: Automation Revolution'
    else:
        return 'Act 3: Fragmentation Era'

df['era'] = [classify_era(y) for y in df.index]

print("✅ Master dataset created with derived metrics")
print("\n📋 Raw Data Preview:")
df[['spending_millions', 'growth_pct', 'programmatic_pct', 'in_app_pct', 'youtube_time_pct']].head(10)

In [ ]:
print("📐 Derived Metrics Preview:")
df[['programmatic_dollars', 'in_app_dollars', 'youtube_implied_opportunity', 'non_youtube_opportunity']].dropna().head(10)

---

# 🎬 ACT 1: The Money Explosion

## From $64 Million to $145 Billion — A 2,268x Journey

The mobile video advertising market exploded from a $64 million experiment in 2011 to a $145 billion industry by 2029. This represents one of the fastest-growing segments in advertising history.

In [ ]:
# VISUALIZATION 1: THE MONEY EXPLOSION
fig, ax = plt.subplots(figsize=(14, 8))

years_plot = df.index.tolist()
spending_plot = df['spending_millions'].values

ax.fill_between(years_plot, spending_plot, alpha=0.3, color=COLORS['primary'])
ax.plot(years_plot, spending_plot, color=COLORS['primary'], linewidth=3, marker='o', markersize=6)

# Era shading
ax.axvspan(2011, 2016.5, alpha=0.1, color=COLORS['warning'], label='Act 1: Wild West')
ax.axvspan(2016.5, 2022.5, alpha=0.1, color=COLORS['purple'], label='Act 2: Automation Revolution')
ax.axvspan(2022.5, 2029, alpha=0.1, color=COLORS['green'], label='Act 3: Fragmentation Era')

# Key milestones
milestones = [
    (2011, 63.8, '$64M\nBirth'),
    (2017, 9800, '$9.8B\nProgrammatic\nTakeover'),
    (2020, 27700, '$27.7B\nCOVID\nAcceleration'),
    (2025, 86000, '$86B\n94% Automated'),
    (2029, 144700, '$145B\n2,268x Growth')
]

for year, val, label in milestones:
    ax.annotate(label, xy=(year, val), xytext=(0, 20),
                textcoords='offset points', ha='center', fontsize=9,
                color=COLORS['warning'],
                arrowprops=dict(arrowstyle='->', color=COLORS['warning'], lw=1.5))

ax.set_xlabel('Year', fontsize=12, fontweight='bold')
ax.set_ylabel('Mobile Video Ad Spending ($ Millions)', fontsize=12, fontweight='bold')
ax.set_title('THE MONEY EXPLOSION\n$64 Million → $145 Billion in 18 Years', 
             fontsize=18, fontweight='bold', color=COLORS['primary'], pad=20)

ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f'${x/1000:.0f}B' if x >= 1000 else f'${x:.0f}M'))
ax.set_xlim(2010.5, 2029.5)
ax.legend(loc='upper left', fontsize=10)

ax.text(0.98, 0.02, '2,268x\nGROWTH', transform=ax.transAxes, fontsize=36, 
        fontweight='bold', color=COLORS['secondary'], alpha=0.3,
        ha='right', va='bottom')

plt.tight_layout()
plt.savefig('01_money_explosion.png', dpi=150, bbox_inches='tight', facecolor='#0a0a0a')
plt.show()

print(f"\n📈 Key Stat: {df.loc[2029, 'spending_millions']/df.loc[2011, 'spending_millions']:.0f}x growth from 2011 to 2029")

---

# 🎬 ACT 2: The Automation Takeover

## By 2025, $81 Billion Will Be Bought Without Human Negotiation

The rise of programmatic advertising fundamentally changed how mobile video ads are bought and sold. What started as 2.6% of the market in 2013 became 94% by 2025 — a complete inversion that made human media buyers effectively obsolete in this channel.

In [ ]:
# VISUALIZATION 2: AUTOMATION TAKEOVER
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

prog_years = df[df['programmatic_pct'].notna()].index.tolist()
prog_data = df.loc[prog_years]

# LEFT: Stacked Area - Programmatic vs Manual $
ax1.stackplot(prog_years, 
              [prog_data['programmatic_dollars'].values, prog_data['manual_dollars'].values],
              labels=['Programmatic (Machine-Bought)', 'Manual (Human-Bought)'],
              colors=[COLORS['primary'], COLORS['secondary']],
              alpha=0.8)

ax1.set_xlabel('Year', fontsize=12, fontweight='bold')
ax1.set_ylabel('Ad Spending ($ Millions)', fontsize=12, fontweight='bold')
ax1.set_title('THE AUTOMATION DOLLAR WAVE\nHow Machines Took Over Ad Buying', 
              fontsize=14, fontweight='bold', color=COLORS['primary'])
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f'${x/1000:.0f}B'))
ax1.legend(loc='upper left')

val_2025_prog = df.loc[2025, 'programmatic_dollars']
ax1.annotate(f'$80.7B\nMachine-Bought', xy=(2025, val_2025_prog/2), 
             fontsize=11, ha='center', color='white', fontweight='bold')

# RIGHT: Programmatic Percentage
ax2.fill_between(prog_years, prog_data['programmatic_pct'].values, alpha=0.4, color=COLORS['primary'])
ax2.plot(prog_years, prog_data['programmatic_pct'].values, 
         color=COLORS['primary'], linewidth=3, marker='s', markersize=8)

ax2.axhline(y=90, color=COLORS['secondary'], linestyle='--', linewidth=2, alpha=0.7)
ax2.text(2027.5, 91.5, 'Human Buyers\nEffectively Extinct', fontsize=10, 
         color=COLORS['secondary'], ha='center')

ax2.set_xlabel('Year', fontsize=12, fontweight='bold')
ax2.set_ylabel('Programmatic Share (%)', fontsize=12, fontweight='bold')
ax2.set_title('SPEED OF HUMAN DISPLACEMENT\nProgrammatic % of Mobile Video', 
              fontsize=14, fontweight='bold', color=COLORS['primary'])
ax2.set_ylim(0, 100)

ax2.annotate('2.6%\n"Experiment"', xy=(2013, 2.6), xytext=(2013, 20),
             ha='center', fontsize=9, color=COLORS['warning'],
             arrowprops=dict(arrowstyle='->', color=COLORS['warning']))
ax2.annotate('94%\n"Total\nDomination"', xy=(2025, 93.8), xytext=(2025, 75),
             ha='center', fontsize=9, color=COLORS['warning'],
             arrowprops=dict(arrowstyle='->', color=COLORS['warning']))

plt.tight_layout()
plt.savefig('02_automation_takeover.png', dpi=150, bbox_inches='tight', facecolor='#0a0a0a')
plt.show()

print(f"\n🤖 By 2025: ${df.loc[2025, 'programmatic_dollars']/1000:.1f}B bought programmatically")
print(f"👤 By 2025: Only ${df.loc[2025, 'manual_dollars']/1000:.1f}B bought by humans")

---

# 🎬 ACT 3: The Death of Mobile Web

## 97% of Mobile Video Ads Now Live in Apps — The Browser Lost

A quiet revolution happened: mobile video advertising completely abandoned the mobile web browser. By 2027, less than 3% of a $117 billion market runs on mobile web. If you're not in an app, you don't exist.

In [ ]:
# VISUALIZATION 3: DEATH OF MOBILE WEB
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

app_years = df[df['in_app_pct'].notna()].index.tolist()
app_data = df.loc[app_years]

# LEFT: Stacked Area - In-App vs Mobile Web $
ax1.stackplot(app_years,
              [app_data['in_app_dollars'].values, app_data['mobile_web_dollars'].values],
              labels=['In-App (Apps Won)', 'Mobile Web (Dying)'],
              colors=[COLORS['green'], COLORS['secondary']],
              alpha=0.8)

ax1.set_xlabel('Year', fontsize=12, fontweight='bold')
ax1.set_ylabel('Ad Spending ($ Millions)', fontsize=12, fontweight='bold')
ax1.set_title('THE APP DOMINANCE\nWhere Mobile Video Ad Dollars Actually Live', 
              fontsize=14, fontweight='bold', color=COLORS['green'])
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f'${x/1000:.0f}B'))
ax1.legend(loc='upper left')

# RIGHT: Mobile Web's Collapse
mobile_web_pct = 100 - app_data['in_app_pct'].values

ax2.fill_between(app_years, mobile_web_pct, alpha=0.4, color=COLORS['secondary'])
ax2.plot(app_years, mobile_web_pct, color=COLORS['secondary'], linewidth=3, marker='o', markersize=8)

ax2.axhline(y=5, color=COLORS['warning'], linestyle='--', linewidth=2, alpha=0.7)
ax2.text(2024, 6.5, '"Effectively Dead" Threshold', fontsize=10, color=COLORS['warning'])

ax2.set_xlabel('Year', fontsize=12, fontweight='bold')
ax2.set_ylabel('Mobile Web Share (%)', fontsize=12, fontweight='bold')
ax2.set_title('THE DEATH OF MOBILE WEB VIDEO\nBrowser Share Collapse', 
              fontsize=14, fontweight='bold', color=COLORS['secondary'])
ax2.set_ylim(0, 30)

ax2.annotate('27.4%\n"Fighting"', xy=(2017, 27.4), xytext=(2017, 25),
             ha='center', fontsize=9, color='white')
ax2.annotate('2.7%\n"Dead"', xy=(2027, 2.7), xytext=(2027, 10),
             ha='center', fontsize=9, color=COLORS['warning'],
             arrowprops=dict(arrowstyle='->', color=COLORS['warning']))

plt.tight_layout()
plt.savefig('04_death_of_mobile_web.png', dpi=150, bbox_inches='tight', facecolor='#0a0a0a')
plt.show()

print(f"\n📱 By 2027: ${df.loc[2027, 'in_app_dollars']/1000:.1f}B in apps")
print(f"🌐 By 2027: Only ${df.loc[2027, 'mobile_web_dollars']/1000:.1f}B on mobile web")
print(f"💀 Mobile web share: {100 - df.loc[2027, 'in_app_pct']:.1f}% — effectively dead")

---

# 🎬 THE CENTRAL INSIGHT: The YouTube Paradox

## The Money Flooded In. YouTube's Attention Leaked Out.

This is the most counter-intuitive finding — and the insight that **only emerges when combining all five datasets**.

While mobile video ad spending grew **6x** from 2017 to 2023, YouTube's share of mobile video attention **collapsed by 50%** — from 42% to 21%.

**Where did the attention go?** TikTok. Instagram Reels. Streaming apps.

In [ ]:
# VISUALIZATION 4: THE YOUTUBE PARADOX (KEY INSIGHT)
fig, ax1 = plt.subplots(figsize=(14, 8))

yt_years = df[df['youtube_time_pct'].notna()].index.tolist()
yt_data = df.loc[yt_years]

# Primary axis: Total Spending (bars)
bars = ax1.bar(yt_years, yt_data['spending_millions'].values/1000, 
               color=COLORS['primary'], alpha=0.6, label='Total Mobile Video Ad Spend')
ax1.set_xlabel('Year', fontsize=12, fontweight='bold')
ax1.set_ylabel('Total Spending ($ Billions)', fontsize=12, fontweight='bold', color=COLORS['primary'])
ax1.tick_params(axis='y', labelcolor=COLORS['primary'])

# Secondary axis: YouTube Time Share (line)
ax2 = ax1.twinx()
ax2.plot(yt_years, yt_data['youtube_time_pct'].values, 
         color=COLORS['secondary'], linewidth=4, marker='o', markersize=10,
         label="YouTube's Share of Mobile Video Time")
ax2.set_ylabel('YouTube Time Share (%)', fontsize=12, fontweight='bold', color=COLORS['secondary'])
ax2.tick_params(axis='y', labelcolor=COLORS['secondary'])
ax2.set_ylim(15, 50)

# The paradox zone
ax1.axvspan(2017, 2023, alpha=0.1, color=COLORS['warning'])
ax1.text(2020, 95, 'THE PARADOX ZONE\nMoney up 6x, YouTube attention down 50%', 
         ha='center', fontsize=11, color=COLORS['warning'], fontweight='bold',
         bbox=dict(boxstyle='round', facecolor='#0a0a0a', edgecolor=COLORS['warning']))

# Annotations
ax2.annotate('41.9%\nYouTube\nDominant', xy=(2017, 41.9), xytext=(2016, 46),
             fontsize=10, color=COLORS['secondary'], fontweight='bold',
             arrowprops=dict(arrowstyle='->', color=COLORS['secondary']))
ax2.annotate('21%\nAttention\nHalved', xy=(2023, 21), xytext=(2024, 28),
             fontsize=10, color=COLORS['secondary'], fontweight='bold',
             arrowprops=dict(arrowstyle='->', color=COLORS['secondary']))

# TikTok emergence
ax1.axvline(x=2020, color=COLORS['pink'], linestyle=':', linewidth=2, alpha=0.7)
ax1.text(2020.1, 10, 'TikTok\nExplodes', fontsize=9, color=COLORS['pink'], rotation=90, va='bottom')

ax1.set_title('THE YOUTUBE PARADOX\nMoney Concentrated, Attention Fragmented', 
              fontsize=18, fontweight='bold', color=COLORS['warning'], pad=20)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

plt.tight_layout()
plt.savefig('05_youtube_paradox.png', dpi=150, bbox_inches='tight', facecolor='#0a0a0a')
plt.show()

spend_growth = df.loc[2023, 'spending_millions'] / df.loc[2017, 'spending_millions']
yt_decline = df.loc[2023, 'youtube_time_pct'] / df.loc[2017, 'youtube_time_pct']
print(f"\n🎯 THE PARADOX IN NUMBERS:")
print(f"   • Spending grew {spend_growth:.1f}x (2017-2023)")
print(f"   • YouTube attention fell to {yt_decline:.1%} of 2017 levels")

In [ ]:
# VISUALIZATION 5: WHERE THE ATTENTION FLOWS
fig, ax = plt.subplots(figsize=(14, 7))

yt_data = df[df['youtube_time_pct'].notna()]
yt_years = yt_data.index.tolist()

ax.stackplot(yt_years,
             [yt_data['youtube_implied_opportunity'].values, 
              yt_data['non_youtube_opportunity'].values],
             labels=["YouTube's Implied Share", 'TikTok/Reels/Streaming Apps'],
             colors=[COLORS['secondary'], COLORS['purple']],
             alpha=0.8)

ax.set_xlabel('Year', fontsize=12, fontweight='bold')
ax.set_ylabel('Implied Ad Opportunity ($ Millions)', fontsize=12, fontweight='bold')
ax.set_title('WHERE THE ATTENTION (AND MONEY) FLOWS\nYouTube vs The New Challengers', 
             fontsize=14, fontweight='bold', color=COLORS['purple'])
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f'${x/1000:.0f}B'))
ax.legend(loc='upper left')

yt_2023 = df.loc[2023, 'youtube_implied_opportunity']
non_yt_2023 = df.loc[2023, 'non_youtube_opportunity']
ax.annotate(f'${non_yt_2023/1000:.1f}B\nNon-YouTube', xy=(2023, yt_2023 + non_yt_2023/2),
            fontsize=11, ha='center', color='white', fontweight='bold')

plt.tight_layout()
plt.savefig('06_attention_flows.png', dpi=150, bbox_inches='tight', facecolor='#0a0a0a')
plt.show()

print(f"\n📊 2023 YouTube implied opportunity: ${df.loc[2023, 'youtube_implied_opportunity']/1000:.1f}B")
print(f"📊 2023 Non-YouTube opportunity: ${df.loc[2023, 'non_youtube_opportunity']/1000:.1f}B")

---

# 📊 Master Dashboard: All Five Metrics

In [ ]:
# VISUALIZATION 6: MASTER DASHBOARD
fig = plt.figure(figsize=(18, 14))
gs = GridSpec(3, 2, figure=fig, hspace=0.35, wspace=0.25)

# 1. Total Spending & Growth Rate
ax1 = fig.add_subplot(gs[0, 0])
ax1b = ax1.twinx()
ax1.bar(df.index, df['spending_millions']/1000, color=COLORS['primary'], alpha=0.7)
ax1b.plot(df.index, df['growth_pct'], color=COLORS['secondary'], linewidth=2, marker='o', markersize=4)
ax1.set_title('1️⃣ THE S-CURVE: Spending & Growth Rate', fontsize=12, fontweight='bold', color='white')
ax1.set_ylabel('Spending ($B)', color=COLORS['primary'])
ax1b.set_ylabel('Growth Rate (%)', color=COLORS['secondary'])
ax1b.set_ylim(0, 300)

# 2. Programmatic Percentage
ax2 = fig.add_subplot(gs[0, 1])
prog_data = df[df['programmatic_pct'].notna()]
ax2.fill_between(prog_data.index, prog_data['programmatic_pct'], alpha=0.5, color=COLORS['purple'])
ax2.plot(prog_data.index, prog_data['programmatic_pct'], color=COLORS['purple'], linewidth=3, marker='s')
ax2.set_title('2️⃣ AUTOMATION TAKEOVER: Programmatic %', fontsize=12, fontweight='bold', color='white')
ax2.set_ylabel('Programmatic %')
ax2.set_ylim(0, 100)
ax2.axhline(y=90, color=COLORS['warning'], linestyle='--', alpha=0.5)

# 3. In-App Percentage
ax3 = fig.add_subplot(gs[1, 0])
app_data = df[df['in_app_pct'].notna()]
ax3.fill_between(app_data.index, app_data['in_app_pct'], alpha=0.5, color=COLORS['green'])
ax3.plot(app_data.index, app_data['in_app_pct'], color=COLORS['green'], linewidth=3, marker='o')
ax3.set_title('3️⃣ APP DOMINANCE: In-App %', fontsize=12, fontweight='bold', color='white')
ax3.set_ylabel('In-App %')
ax3.set_ylim(70, 100)

# 4. YouTube Time Share
ax4 = fig.add_subplot(gs[1, 1])
yt_data = df[df['youtube_time_pct'].notna()]
ax4.fill_between(yt_data.index, yt_data['youtube_time_pct'], alpha=0.5, color=COLORS['secondary'])
ax4.plot(yt_data.index, yt_data['youtube_time_pct'], color=COLORS['secondary'], linewidth=3, marker='o')
ax4.set_title('4️⃣ YOUTUBE PARADOX: Time Share %', fontsize=12, fontweight='bold', color='white')
ax4.set_ylabel('YouTube Time Share %')
ax4.set_ylim(15, 50)

# 5. $ per YouTube Attention Point
ax5 = fig.add_subplot(gs[2, :])
attention_data = df[df['dollars_per_yt_attention_point'].notna()]
bars = ax5.bar(attention_data.index, attention_data['dollars_per_yt_attention_point'], 
               color=COLORS['orange'], alpha=0.8)
ax5.set_title('5️⃣ THE ATTENTION ARBITRAGE: $ Spent per % of YouTube Attention', 
              fontsize=12, fontweight='bold', color='white')
ax5.set_ylabel('$ Million per Attention Point')
ax5.set_xlabel('Year')

for bar, val in zip(bars, attention_data['dollars_per_yt_attention_point']):
    ax5.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50, 
             f'${val/1000:.1f}B', ha='center', fontsize=8, color='white')

fig.suptitle('THE MOBILE VIDEO AD LANDSCAPE: 5 METRICS THAT TELL THE STORY', 
             fontsize=18, fontweight='bold', color=COLORS['primary'], y=0.98)

plt.savefig('07_master_dashboard.png', dpi=150, bbox_inches='tight', facecolor='#0a0a0a')
plt.show()

---

# 🔥 Year-over-Year Changes Heatmap

In [ ]:
# VISUALIZATION 7: YoY HEATMAP
fig, ax = plt.subplots(figsize=(16, 8))

yoy_df = pd.DataFrame({
    'Spending': df['spending_millions'].pct_change() * 100,
    'Programmatic %': df['programmatic_pct'].diff(),
    'In-App %': df['in_app_pct'].diff(),
    'YouTube Time %': df['youtube_time_pct'].diff()
})

yoy_df = yoy_df.dropna(how='all')

sns.heatmap(yoy_df.T, annot=True, fmt='.1f', cmap='RdYlGn', center=0,
            linewidths=0.5, ax=ax, cbar_kws={'label': 'YoY Change'})

ax.set_title('YEAR-OVER-YEAR CHANGES HEATMAP\nGreen = Growth/Increase | Red = Decline', 
             fontsize=14, fontweight='bold', color='white')
ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Metric', fontsize=12)

plt.tight_layout()
plt.savefig('08_yoy_heatmap.png', dpi=150, bbox_inches='tight', facecolor='#0a0a0a')
plt.show()

---

# 🎬 The Three-Act Story Visualization

In [ ]:
# VISUALIZATION 8: THREE ACT STORY
fig, ax = plt.subplots(figsize=(16, 9))

ax.axvspan(2011, 2016.5, alpha=0.15, color=COLORS['warning'], label='Act 1: Wild West')
ax.axvspan(2016.5, 2022.5, alpha=0.15, color=COLORS['purple'], label='Act 2: Automation Revolution')
ax.axvspan(2022.5, 2029, alpha=0.15, color=COLORS['green'], label='Act 3: Fragmentation Era')

ax.fill_between(df.index, df['spending_millions']/1000, alpha=0.5, color=COLORS['primary'])
ax.plot(df.index, df['spending_millions']/1000, color=COLORS['primary'], linewidth=3)

era_info = [
    (2013.5, 'ACT 1: WILD WEST\n(2011-2016)', 
     '• 100%+ YoY growth\n• Manual ad buying\n• YouTube dominates (40%+)', COLORS['warning']),
    (2019.5, 'ACT 2: AUTOMATION\nREVOLUTION (2017-2022)', 
     '• Programmatic: 40%→90%\n• Apps devour mobile web\n• TikTok emerges\n• COVID accelerates', COLORS['purple']),
    (2026, 'ACT 3: FRAGMENTATION\nERA (2023-2029)', 
     '• 10-15% YoY growth\n• 95%+ programmatic\n• 97%+ in-app\n• $145B chasing\n  fragmented attention', COLORS['green'])
]

for x, title, details, color in era_info:
    ax.text(x, 135, title, fontsize=12, fontweight='bold', color=color, ha='center',
            bbox=dict(boxstyle='round', facecolor='#0a0a0a', edgecolor=color, alpha=0.9))
    ax.text(x, 115, details, fontsize=9, color='white', ha='center', va='top', linespacing=1.5)

events = [
    (2012, 0.246, 'Facebook\nIPO'),
    (2017, 9.8, 'Programmatic\nHits 80%'),
    (2020, 27.7, 'COVID\nBoom'),
    (2021, 40.8, 'TikTok\n$1B ads'),
]

for x, y, label in events:
    ax.annotate(label, xy=(x, y), xytext=(0, -40), textcoords='offset points',
                ha='center', fontsize=8, color=COLORS['secondary'],
                arrowprops=dict(arrowstyle='->', color=COLORS['secondary']))

ax.set_xlabel('Year', fontsize=12, fontweight='bold')
ax.set_ylabel('Mobile Video Ad Spending ($ Billions)', fontsize=12, fontweight='bold')
ax.set_title('THE THREE-ACT STORY OF MOBILE VIDEO ADVERTISING', 
             fontsize=18, fontweight='bold', color='white', pad=20)
ax.set_xlim(2010.5, 2029.5)
ax.set_ylim(0, 160)
ax.legend(loc='lower right')

plt.tight_layout()
plt.savefig('09_three_act_story.png', dpi=150, bbox_inches='tight', facecolor='#0a0a0a')
plt.show()

---

# 📌 Executive Summary Visual

In [ ]:
# VISUALIZATION 9: EXECUTIVE SUMMARY
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('THE $145 BILLION PARADOX: EXECUTIVE SUMMARY', 
             fontsize=20, fontweight='bold', color=COLORS['primary'], y=1.02)

metrics = [
    ('💰 Money Explosion', f'{df.loc[2029, "spending_millions"]/df.loc[2011, "spending_millions"]:.0f}x', 
     'Growth 2011-2029', COLORS['primary']),
    ('🤖 Automation', f'{df.loc[2025, "programmatic_pct"]:.0f}%', 
     'Programmatic by 2025', COLORS['purple']),
    ('📱 App Dominance', f'{df.loc[2027, "in_app_pct"]:.0f}%', 
     'In-App by 2027', COLORS['green']),
    ('📺 YouTube Decline', f'-{(1 - df.loc[2023, "youtube_time_pct"]/df.loc[2017, "youtube_time_pct"])*100:.0f}%', 
     'Attention 2017-2023', COLORS['secondary'])
]

for ax, (title, value, subtitle, color) in zip(axes.flatten(), metrics):
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 10)
    ax.set_aspect('equal')
    ax.axis('off')
    
    circle = plt.Circle((5, 5), 4, color=color, alpha=0.2)
    ax.add_patch(circle)
    
    ax.text(5, 8, title, fontsize=14, fontweight='bold', ha='center', color='white')
    ax.text(5, 5, value, fontsize=36, fontweight='bold', ha='center', va='center', color=color)
    ax.text(5, 2, subtitle, fontsize=11, ha='center', color='#888888')

plt.tight_layout()
plt.savefig('10_executive_summary.png', dpi=150, bbox_inches='tight', facecolor='#0a0a0a')
plt.show()

---

# 📋 Final Summary Statistics

In [ ]:
print("="*70)
print("📊 THE $145 BILLION PARADOX: KEY STATISTICS")
print("="*70)

print("\n💰 THE MONEY EXPLOSION")
print(f"   • 2011: ${df.loc[2011, 'spending_millions']:.1f}M")
print(f"   • 2029: ${df.loc[2029, 'spending_millions']/1000:.1f}B")
print(f"   • Growth: {df.loc[2029, 'spending_millions']/df.loc[2011, 'spending_millions']:.0f}x in 18 years")

print("\n🤖 THE AUTOMATION TAKEOVER")
print(f"   • 2013: {df.loc[2013, 'programmatic_pct']:.1f}% programmatic")
print(f"   • 2025: {df.loc[2025, 'programmatic_pct']:.1f}% programmatic")
print(f"   • 2025 Programmatic $: ${df.loc[2025, 'programmatic_dollars']/1000:.1f}B")

print("\n📱 THE APP DOMINANCE")
print(f"   • 2017: {df.loc[2017, 'in_app_pct']:.1f}% in-app")
print(f"   • 2027: {df.loc[2027, 'in_app_pct']:.1f}% in-app")
print(f"   • Mobile web by 2027: {100 - df.loc[2027, 'in_app_pct']:.1f}% (effectively dead)")

print("\n📺 THE YOUTUBE PARADOX")
print(f"   • 2017 YouTube time share: {df.loc[2017, 'youtube_time_pct']:.1f}%")
print(f"   • 2023 YouTube time share: {df.loc[2023, 'youtube_time_pct']:.1f}%")
print(f"   • Attention decline: {(1 - df.loc[2023, 'youtube_time_pct']/df.loc[2017, 'youtube_time_pct'])*100:.0f}%")
print(f"   • While spending grew: {df.loc[2023, 'spending_millions']/df.loc[2017, 'spending_millions']:.1f}x")

print("\n" + "="*70)
print("💡 THE ONE-SENTENCE STORY:")
print("="*70)
print('"Mobile video advertising became a $145B automated, app-dominated machine')
print(' — but while the money concentrated, the audience fragmented, and YouTube')
print(' lost half its attention share to TikTok, Reels, and streaming apps."')
print("="*70)

In [ ]:
# Export complete dataset
df.to_csv('mobile_video_ad_analysis_complete.csv')
print("✅ Complete dataset exported to: mobile_video_ad_analysis_complete.csv")

# List generated visualizations
import os
pngs = [f for f in os.listdir('.') if f.endswith('.png')]
print(f"\n📊 Generated {len(pngs)} visualizations:")
for png in sorted(pngs):
    print(f"   • {png}")

---

## 🎬 Conclusion

From 2011 to 2029, mobile video advertising underwent a transformation unprecedented in media history. What started as a $64 million experiment became a **$145 billion automated machine** — growing 2,268x in just 18 years.

Along the way:
- **Programmatic buying eliminated human ad negotiators** (from 2.6% to 94%)
- **Apps crushed mobile web browsers** (97% vs 3%)
- **YouTube lost half its mobile video attention** to TikTok, Instagram Reels, and streaming apps — even as total ad dollars quintupled

This is the paradox of modern digital advertising: **the money concentrated while the audience fragmented**.

---

*Data Source: EMARKETER | Analysis by HackrLife | January 2026*